# Direct GRPO: Live Pairwise Judging with Llama 70B

Train Qwen 2.5 7B policy using GRPO where rewards come from **live pairwise judging** by Llama 70B at each training step (the "direct" RLAIF approach).

For each group of G=4 completions:
- Form all C(4,2)=6 pairs, query both orderings (12 judge calls) to control position bias
- Each completion's reward = its win fraction across comparisons

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets trl peft accelerate openai

In [ ]:
import os

# Set your Together API key here
os.environ["TOGETHER_API_KEY"] = "YOUR_KEY_HERE"  # <-- replace this

In [ ]:
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import combinations

import openai
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

## 2. Configuration

In [ ]:
# ── Models ───────────────────────────────────────────────────────────────
POLICY_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# ── Data ─────────────────────────────────────────────────────────────────
DATA_PATH = "preferences.jsonl"  # upload this to Colab or mount Drive
OUTPUT_DIR = "./grpo_direct"
EVAL_SPLIT = 0.1

# ── GRPO hyperparameters (matched to indirect for fair comparison) ───────
NUM_GENERATIONS = 4       # G: completions per prompt
MAX_COMPLETION_LENGTH = 256
TEMPERATURE = 0.8
BETA = 0.05               # KL penalty coefficient
LEARNING_RATE = 5e-6
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_STEPS = 200

# ── LoRA ─────────────────────────────────────────────────────────────────
LORA_R = 16
LORA_ALPHA = 64

# ── Judge API ────────────────────────────────────────────────────────────
JUDGE_WORKERS = 16         # parallel threads — all prompts judged at once
MAX_RETRIES = 5
BASE_DELAY = 1.0

## 3. Judge API utilities

In [ ]:
client = openai.OpenAI(
    api_key=os.environ["TOGETHER_API_KEY"],
    base_url="https://api.together.xyz/v1",
)

JUDGE_TEMPLATE = """You are an impartial judge. Given an instruction and two responses, decide which response is better.

Consider: accuracy, helpfulness, clarity, and conciseness.

Respond with EXACTLY "A" or "B" (no other text).

Instruction: {instruction}

Response A:
{response_a}

Response B:
{response_b}"""


def api_call_with_retry(fn):
    """Retry an API call with exponential backoff."""
    for attempt in range(MAX_RETRIES):
        try:
            return fn()
        except (openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError) as e:
            if attempt == MAX_RETRIES - 1:
                raise
            delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
            print(f"    [retry {attempt + 1}/{MAX_RETRIES}] {type(e).__name__}, waiting {delay:.1f}s")
            time.sleep(delay)
        except openai.APIStatusError as e:
            if e.status_code >= 500 and attempt < MAX_RETRIES - 1:
                delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
                print(f"    [retry {attempt + 1}/{MAX_RETRIES}] {e.status_code}, waiting {delay:.1f}s")
                time.sleep(delay)
            else:
                raise


def parse_verdict(judgment: str) -> str | None:
    """Extract 'A' or 'B' from the judge response."""
    text = judgment.strip().upper()
    if text in ("A", "B"):
        return text
    for line in judgment.splitlines():
        line = line.strip().upper()
        if line in ("A", "B"):
            return line
        if line.startswith("WINNER:"):
            pick = line.split(":", 1)[1].strip()
            if pick in ("A", "B"):
                return pick
    return None


def judge_single_pair(instruction: str, response_a: str, response_b: str) -> str | None:
    """Query Llama 70B to judge a single ordered pair. Returns 'A' or 'B' or None."""
    prompt = JUDGE_TEMPLATE.format(
        instruction=instruction,
        response_a=response_a,
        response_b=response_b,
    )
    def call():
        return client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.0,
        )
    resp = api_call_with_retry(call)
    return parse_verdict(resp.choices[0].message.content)

### Sanity check: verify judge API works

In [ ]:
# Quick test: judge should pick the good response over the bad one
test_instruction = "What is machine learning?"
test_good = (
    "Machine learning is a subset of artificial intelligence where "
    "systems learn patterns from data to make predictions without "
    "being explicitly programmed."
)
test_bad = "idk lol"

verdict = judge_single_pair(test_instruction, test_good, test_bad)
print(f"Good=A vs Bad=B → verdict: {verdict}")
assert verdict == "A", f"Expected 'A', got '{verdict}'"

verdict_flipped = judge_single_pair(test_instruction, test_bad, test_good)
print(f"Bad=A vs Good=B → verdict: {verdict_flipped}")
assert verdict_flipped == "B", f"Expected 'B', got '{verdict_flipped}'"

# Test win fraction computation with 3 dummy completions
win_fracs = compute_win_fractions(
    test_instruction,
    [test_good, test_bad, "Machine learning uses data to learn."],
    workers=4,
)
print(f"\nWin fractions: {win_fracs}")
print(f"Best completion index: {win_fracs.index(max(win_fracs))}")
print("\nAll checks passed!")

## 4. Win fraction computation

For G completions, form all C(G,2) unordered pairs. Query both orderings per pair to control position bias. Each completion's reward = wins / total comparisons it participated in.

In [ ]:
def compute_win_fractions(instruction: str, completions: list[str], workers: int) -> list[float]:
    """Compute win fraction for each completion via pairwise judging."""
    G = len(completions)
    pairs = list(combinations(range(G), 2))
    wins = [0.0] * G
    total = [0] * G

    # 2 orderings per unordered pair
    tasks = []
    for i, j in pairs:
        tasks.append((i, j))
        tasks.append((j, i))

    results = {}

    def run_judge(a_idx, b_idx):
        verdict = judge_single_pair(instruction, completions[a_idx], completions[b_idx])
        return (a_idx, b_idx, verdict)

    # Don't create a new pool — caller passes futures into a shared pool
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(run_judge, a_idx, b_idx): (a_idx, b_idx)
            for a_idx, b_idx in tasks
        }
        for future in as_completed(futures):
            a_idx, b_idx, verdict = future.result()
            results[(a_idx, b_idx)] = verdict

    for i, j in pairs:
        v1 = results.get((i, j))
        if v1 == "A":
            wins[i] += 1
        elif v1 == "B":
            wins[j] += 1

        v2 = results.get((j, i))
        if v2 == "A":
            wins[j] += 1
        elif v2 == "B":
            wins[i] += 1

        total[i] += 2
        total[j] += 2

    return [wins[k] / total[k] if total[k] > 0 else 0.0 for k in range(G)]


def compute_win_fractions_batch(instructions: list[str], completions_per_prompt: list[list[str]], workers: int) -> list[list[float]]:
    """Compute win fractions for multiple prompts in parallel using a single thread pool."""
    # Build all tasks across all prompts
    all_tasks = []  # (prompt_idx, a_idx, b_idx)
    for p_idx, comps in enumerate(completions_per_prompt):
        G = len(comps)
        pairs = list(combinations(range(G), 2))
        for i, j in pairs:
            all_tasks.append((p_idx, i, j))
            all_tasks.append((p_idx, j, i))

    # Fire all judge calls at once
    results = {}

    def run_judge(p_idx, a_idx, b_idx):
        verdict = judge_single_pair(
            instructions[p_idx],
            completions_per_prompt[p_idx][a_idx],
            completions_per_prompt[p_idx][b_idx],
        )
        return (p_idx, a_idx, b_idx, verdict)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(run_judge, p_idx, a_idx, b_idx): (p_idx, a_idx, b_idx)
            for p_idx, a_idx, b_idx in all_tasks
        }
        for future in as_completed(futures):
            p_idx, a_idx, b_idx, verdict = future.result()
            results[(p_idx, a_idx, b_idx)] = verdict

    # Tally per prompt
    all_win_fracs = []
    for p_idx, comps in enumerate(completions_per_prompt):
        G = len(comps)
        pairs = list(combinations(range(G), 2))
        wins = [0.0] * G
        total = [0] * G

        for i, j in pairs:
            v1 = results.get((p_idx, i, j))
            if v1 == "A":
                wins[i] += 1
            elif v1 == "B":
                wins[j] += 1

            v2 = results.get((p_idx, j, i))
            if v2 == "A":
                wins[j] += 1
            elif v2 == "B":
                wins[i] += 1

            total[i] += 2
            total[j] += 2

        all_win_fracs.append([wins[k] / total[k] if total[k] > 0 else 0.0 for k in range(G)])

    return all_win_fracs

## 5. Reward function wrapper

In [ ]:
def make_judge_reward_fn(judge_workers: int):
    """Create a reward function that uses live pairwise judging.

    GRPOTrainer calls reward_fn(prompts, completions) where prompts come in
    groups of G (num_generations) per original prompt.

    All judge calls across all prompts are fired in parallel.
    """
    def reward_fn(prompts, completions, **kwargs):
        batch_size = len(prompts)

        # Detect group size by counting consecutive identical prompts
        G = 1
        while G < batch_size and prompts[G] == prompts[0]:
            G += 1

        # Collect all prompts and their completion groups
        instructions = []
        completions_per_prompt = []
        for start in range(0, batch_size, G):
            instruction = prompts[start]
            try:
                if "<|im_start|>user\n" in instruction:
                    instruction = instruction.split("<|im_start|>user\n")[1].split("<|im_end|>")[0].strip()
            except (IndexError, AttributeError):
                pass
            instructions.append(instruction)
            completions_per_prompt.append(completions[start:start + G])

        # Fire ALL judge calls across ALL prompts in one thread pool
        all_win_fracs = compute_win_fractions_batch(
            instructions, completions_per_prompt, workers=judge_workers,
        )

        # Flatten back to per-completion rewards
        rewards = []
        for win_fracs in all_win_fracs:
            rewards.extend(win_fracs)

        return rewards

    return reward_fn

## 6. Load data

In [ ]:
prompts = []
seen = set()
with open(DATA_PATH) as f:
    for line in f:
        row = json.loads(line)
        inst = row["instruction"]
        if inst not in seen:
            seen.add(inst)
            prompts.append([{"role": "user", "content": inst}])

dataset = Dataset.from_dict({"prompt": prompts})
split = dataset.train_test_split(test_size=EVAL_SPLIT, seed=42)
print(f"Train: {len(split['train'])} | Eval: {len(split['test'])} prompts")

## 7. Initialize trainer

In [ ]:
# ── Policy tokenizer ─────────────────────────────────────────────────────
policy_tokenizer = AutoTokenizer.from_pretrained(POLICY_MODEL)
if policy_tokenizer.pad_token is None:
    policy_tokenizer.pad_token = policy_tokenizer.eos_token
policy_tokenizer.padding_side = "left"

# ── Reward function ──────────────────────────────────────────────────────
reward_fn = make_judge_reward_fn(judge_workers=JUDGE_WORKERS)

judge_calls_per_step = BATCH_SIZE * NUM_GENERATIONS * (NUM_GENERATIONS - 1)
print(f"Judge calls per training step: {judge_calls_per_step}")
print(f"Judge calls per step (with grad accum): {judge_calls_per_step * GRAD_ACCUM}")

# ── LoRA config ──────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# ── GRPO config ──────────────────────────────────────────────────────────
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    beta=BETA,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=20,
    logging_steps=10,
    save_steps=50,
    eval_strategy="no",
    save_total_limit=5,
    bf16=use_bf16,
    fp16=not use_bf16 and torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

# ── Trainer ──────────────────────────────────────────────────────────────
trainer = GRPOTrainer(
    model=POLICY_MODEL,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=lora_config,
    processing_class=policy_tokenizer,
)

print(f"\nReady: policy={POLICY_MODEL}, G={NUM_GENERATIONS}, β={BETA}")

## 8. Train

In [ ]:
trainer.train()

## 9. Save model

In [ ]:
trainer.save_model(OUTPUT_DIR)
policy_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 10. (Optional) Save to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./grpo_direct /content/drive/MyDrive/grpo_direct